# Nurse Rostering (INRC-II): exato, GRASP, SA e Fix-and-Optimize

Antes de rodar, envie pelo painel de arquivos (ícone de pasta, à esquerda):
* `nurse_rostering.zip` (o projeto)
* `testdatasets_json.zip` (instâncias de teste do INRC-II)
* os outros pacotes JSON que tiverem: `datasets_json.zip`, `hidden-JSON.zip`, `late-extended-dataset.zip`

Os nomes podem ter prefixos (por exemplo `1794..._datasets_json.zip`); a primeira célula pega todos os `.zip` da pasta.

In [ ]:
!pip -q install highspy swiglpk numba
!unzip -q -o nurse_rostering.zip -d .
!python nurse_rostering/organizar_dados.py dados $(ls *.zip | grep -v nurse_rostering)
%cd nurse_rostering

## 1. Testes de correção
Deve terminar com `TUDO CERTO` (leva cerca de 1 minuto).

In [ ]:
!python testes.py ../dados

## 2. Rodar cada método numa instância
Argumentos: `pasta historico semanas tempo_em_segundos`.

In [ ]:
!python exato.py ../dados/n005w4 0 1,2,3,3 60

In [ ]:
!python grasp.py ../dados/n005w4 0 1,2,3,3 60
!python sa.py ../dados/n005w4 0 1,2,3,3 60

In [ ]:
!python fo.py ../dados/n021w4 0 5,4,1,2 120     # SA + Fix-and-Optimize

## 3. Experimentos
Roda exato (HiGHS), GRASP, SA e SA + Fix-and-Optimize e grava um CSV. Tempo total ≈ instâncias × (1 + 3 × sementes) × tempo.
Nas 40 instâncias de `referencias_literatura.csv`, o CSV traz também o melhor valor publicado (`ub_literatura`) e o gap contra ele (`gap_ub_lit_%`).

In [ ]:
# teste rapido (3 instancias de teste, 60 s; exato HiGHS, GRASP, SA e SA+F&O):
!python experimentos.py --dados ../dados --instancias instancias.txt --tempo 60 --sementes 1 --saida resultados_teste.csv

# conjunto final (8 instancias da literatura, 300 s, 3 sementes, so as heuristicas): no Colab (2 nucleos) leva ~3 h.
# Grava resultados/v3_*.csv e retoma de onde parou se a sessao cair. No PC, use --processos 6 (~1 h).
# !python rodar_experimentos.py --processos 2

In [ ]:
import pandas as pd
arq = "resultados_teste.csv"          # troque para "resultados_final.csv"
r = pd.read_csv(arq)
display(r)
tab = r.groupby(["instancia", "metodo"]).agg(custo_medio=("custo", "mean"), melhor=("custo", "min"),
                                             gap_lit_medio=("gap_ub_lit_%", "mean"),
                                             tempo_s=("tempo_s", "mean")).unstack("metodo")
display(tab)

In [ ]:
import json, matplotlib.pyplot as plt
conv = json.load(open(arq.replace(".csv", "_convergencia.json")))
inst = sorted({k.split("|")[0] for k in conv})[-1]          # troque pelo nome da instancia desejada
for k, h in conv.items():
    if k.startswith(inst) and h:
        t, c = zip(*h)
        plt.step(t, c, where="post", label=k.split("|")[1])
plt.xlabel("tempo (s)"); plt.ylabel("melhor custo"); plt.title(inst); plt.legend(); plt.show()

## 4. (Opcional) O mesmo modelo no AMPL
Precisa de licença (Community Edition ou AMPL for Academics). O `rodar.run` usa o Gurobi, que devolve o limite inferior. Se a sua licença não incluir o Gurobi, troque para o bloco do HiGHS no início de `rodar.run`. Nesse caso a coluna do limite inferior pode sair vazia, e o limite vem do `exato.py`, que resolve o mesmo modelo.

In [ ]:
!pip -q install amplpy
from amplpy import ampl_notebook
ampl = ampl_notebook(modules=["gurobi", "highs"], license_uuid="COLE-SUA-LICENCA-AQUI")
ampl.cd("/content/nurse_rostering")
ampl.eval("include rodar.run;")            # 9 instancias de teste, 60 s cada  -> resultados_ampl.csv
# ampl.eval("include rodar_final.run;")    # 8 instancias da literatura, 300 s -> resultados_ampl_final.csv
print(open("/content/nurse_rostering/resultados_ampl.csv").read())

## 5. Baixar os resultados

In [ ]:
from google.colab import files
import glob
for f in glob.glob("resultados*.csv"):
    files.download(f)